# From Cellular Automata to Neural Cellular Automata

This notebook introduces the core idea behind Neural Cellular Automata (NCA): complex global context can be achieved when every cell repeatedly applies the same local rule. We first explore handcrafted rules in an interactive Cellular Automata playground. We then replace the handcrafted update with a small neural network, train it to grow a target image, and inspect the resulting dynamics and hidden states. Finally, we compare pretrained NCAs that differ in one training ingredient at a time to see what makes them stable and able to repair damage, and look at an NCA that learned a texture instead of a single image.

The goal is not to obtain the best possible model, but to understand the basic components and mechanisms, as well as the transition from handcrafted updates to learned updates.

## 1. Cellular Automata Playground

Before building our first NCA, explore the basic mechanism shared by both CA and NCA:

$$\text{state} \rightarrow \text{perceive locally} \rightarrow \text{update locally} \rightarrow \text{repeat}$$

The cell below contains an interactive app in which you can draw initial states, load presets, change how cells perceive their neighborhood, and define how they update. Read the following three experiments, then run the app and work through them.

### 1.1 Conway's Game of Life: binary states

Game of Life is a **binary cellular automaton**: every cell is either dead (0) or alive (1). At every generation, all cells apply the same rule simultaneously. A cell first counts the living cells in its eight surrounding positions. It then updates as follows:

- A dead cell becomes alive when it has exactly **3** living neighbors (*birth*).
- A living cell remains alive when it has **2 or 3** living neighbors (*survival*).
- In every other case, the cell is dead in the next generation.

Configure the app with **Perceive → Neighbors** and **Update → Game of Life**. The rule type should be **Life-like (binary)**, with `Birth = 3` and `Survive = 2,3`.

**Tasks**

1. Load a binary preset, such as **Glider**, **Random binary**, or draw your own initial state directly on the canvas. Use **Step** before pressing **Start** and try to predict which cells will change.
2. Change one Birth or Survival value. How does this affect growth, stability, and extinction?
3. Modify entries in the perception kernel. How does it change the behaviour?

### 1.2 Continuous state spaces

Cellular automata do not have to be binary. In the app, cells can also take continuous values between 0 and 1, shown as grayscale intensities. Start with **Perceive → Average** and enable a continuous update such as **Update → Diffusion**. The normalized average keeps perceived values in a convenient range; an unnormalized neighbor sum can quickly become much larger than 1 and saturate a clamped update.

**Tasks**

1. Load **Random continuous**, or clear the grid and draw with a gray brush value.
2. Advance one step at a time. How does diffusion redistribute information across the grid?
3. Try the other continuous update presets and change their formula. The variables are `x` for the current cell state and `p` for its local perception.
4. Compare **Average**, **Laplacian**, and **Sobel-x** perception. What kind of local information does each kernel emphasize?

### 1.3 From pattern formation to image processing

The grid can also be interpreted as an image: every cell is a pixel, and the local rule becomes a repeatedly applied image-processing operation.

**Tasks**

1. Load the **Image** initial-state preset.
2. Keep the rule type continuous and compare the **Average**, **Laplacian**, and **Sobel-x** perception presets.
3. Experiment with diffusion and custom update formulas. Can you smooth, emphasize, or distort structures in the image?
4. Use **Reset** frequently so that you can compare different rules from the same initial image.

In [ ]:
#@title Cellular Automata playground app (run to show it)
from html import escape
from IPython.display import HTML, display

ca_playground = r"""
<!doctype html><html><head><meta charset="utf-8"><style>
*{box-sizing:border-box}body{margin:0;font-family:system-ui,sans-serif;color:#202124}.app{max-width:1100px;margin:auto;padding:14px;border:1px solid #dadce0;border-radius:14px;background:#f8f9fa}.top,.row,.stats{display:flex;gap:8px;align-items:center;flex-wrap:wrap}.top{margin-bottom:12px}.layout{display:grid;grid-template-columns:minmax(360px,1fr) 340px;gap:14px}.canvas-wrap{background:#222;border-radius:10px;padding:8px;display:flex}.canvas-wrap canvas{width:100%;aspect-ratio:1;cursor:crosshair;touch-action:none}.panel{display:flex;flex-direction:column;gap:10px;max-height:680px;overflow:auto}.card{background:#fff;border:1px solid #dadce0;border-radius:10px;padding:11px}.card h3{font-size:15px;margin:0 0 8px}button,select,input{font:inherit;border:1px solid #bdc1c6;border-radius:7px;background:#fff;padding:6px 8px}button{cursor:pointer;font-weight:600}.primary{background:#333;color:#fff!important}.grow{flex:1}.wide{width:100%}label{font-size:13px;font-weight:600}.kernel{display:grid;grid-template-columns:repeat(3,52px);gap:4px;margin:8px 0}.kernel input{width:52px;text-align:center;padding:5px 2px}.small{font-size:12px;color:#5f6368;line-height:1.4}.formula{font-family:monospace;font-size:12px}.stats{justify-content:space-between;margin-top:8px;font-size:13px}.hidden{display:none}.error{color:#b3261e;font-weight:600}.swatch{height:9px;border-radius:5px;background:linear-gradient(90deg,#000,#fff);border:1px solid #aaa}@media(max-width:800px){.layout{grid-template-columns:1fr}.panel{max-height:none}}
</style></head><body><div class="app"><div class="top">
<button id="start" class="primary">▶ Start</button><button id="step">Step</button><button id="reset">Reset</button><button id="clear">Clear</button><label>Speed <input id="speed" type="range" min="1" max="30" value="8"></label></div>
<div class="layout"><div><div class="canvas-wrap"><canvas id="grid" width="640" height="640"></canvas></div><div class="stats"><span id="stats"></span><span id="message">Click or drag to draw.</span></div></div><div class="panel">
<div class="card"><h3>Initial state</h3><div class="row"><select id="initial" class="grow"><option>Single impulse</option><option>Glider</option><option>Random binary</option><option>Random continuous</option><option>Image</option></select><button id="load">Load</button></div><label>Brush value: <span id="brushLabel">1.00</span></label><input id="brush" class="wide" type="range" min="0" max="1" step="0.05" value="1"><div class="swatch"></div><div class="small">Left-click paints this value; right-click erases.</div></div>
<div class="card"><h3>1 · Perceive</h3><div class="row"><label>Preset <select id="perceptionPreset"><option value="neighbors">Neighbors</option><option value="average">Average</option><option value="laplace">Laplacian</option><option value="sobel">Sobel-x</option></select></label></div><div class="small">Weighted sum of the 3×3 neighborhood:</div><div id="kernel" class="kernel"></div><label><input id="normalize" type="checkbox"> Normalize by |weights|</label></div>
<div class="card"><h3>2 · Update</h3><div class="row"><label>Preset <select id="updatePreset"><option value="life">Game of Life</option><option value="diffusion">Diffusion</option><option value="growth">Threshold growth</option><option value="oscillating">Oscillating response</option><option value="tent">Tent map</option><option value="custom">Custom</option></select></label></div><label>Rule type <select id="ruleType"><option value="life">Life-like (binary)</option><option value="formula">Continuous formula</option></select></label><div id="life"><div class="row"><label>Birth <input id="born" value="3" size="7"></label><label>Survive <input id="survive" value="2,3" size="7"></label></div><div class="small">Comma-separated perceived values.</div></div><div id="continuous" class="hidden"><label>New state =</label><input id="formula" class="wide formula" value="x + 0.25 * (p - x)"><div class="small"><b>x</b>: current value · <b>p</b>: perception<br>Functions: clip, abs, sin, cos, exp, min, max</div><label><input id="clamp" type="checkbox" checked> Clamp to [0,1]</label></div></div>
</div></div></div><script>
const N=48,C=document.getElementById('grid'),X=C.getContext('2d');let g=blank(),gen=0,running=false,timer,drawing=false,paintValue=1;const PORTRAIT='Gh4iJCIiHx4eHh0bHBkZHBwaFxcZGBocHB4gIRwcHx8bGxweGxocHB4fHhwdGxgXHSIkISEjIiEhISAhHxsbHB0aFxcXGBkYGhsfHiEhIB8fHx0eHRwcGxobIB8hHxgWHiEiICEnJiIjIyEfHxsZGhwYFxgZGhobGBogHRYXGx0hIyEgHh4bHB8jIh4bHBcWICEjJCMoKCQiICEhHBsaGhoZGBkcHh8eHSEaLVVdRTUlICUkHyAhISIjHx8aHBUXICMnJSUlJSUlIyIfHR8eGxgaGxweHx8eGxRFk5mnyb6fVyghICAhICEgHh8bHhgYHyElJSMmJSUmJiUkJCMhICAhHyAjIx4bLlyvupun1N/gqGU1ISEgISIhIyIgIBgYHiImJicoJycoKCYnJicmIyMiIh4aKDxfp8XO3s20r6eqt2hIMyMhICIkIyAgHxYYHiIlJyonKSooKCcnJSUmJCMiHStklq+tpp6MhLLJr6GOoZZZPiwhICAkIiAeHhgaHiEjJSgoKCorKSgqJiQkIiYgNKDIw7qsqaqjim1/p7KTgoh4SDQiHyMjISAdHRocHSEjJygoKSorKispJSQiIyU0rMa6vr6vo6OgnZZ6YYCgdF55VD0kHiAiIB4bGBcYICMjJSgoKisrKSkoJyMgJC2Qxb+/vbWtpJ+glpiZaEhfg1J9YDopHyIgHx8bGxYWIiQlJiYpKiwrKCYmKCUlHm3Fub24tbCmpqWcmpeQb1dDUmCLfEYwHB4eHhwdHhgXICMnKScpKiopKCYmKCckMZ6ssLe2tpt/mqCjiI9/SldVS0mdilM/FhsbGxsfHRoYISInKy0rKCkqKSkoJycjOqHN08WrpY58c3WQaU9fQEJPTUOMiFFYHhIYGx0dGxkZHyEkKiopJycpJiknKigdSdXj4dvKroqUck9iVzdFTEVKUkB9hkxlLhgZHBsdHBsXHh8jJycmJyYlKCopKCkekObb2tLRzaZ3X0FHS0dEVU5GST1tjEthMiEXGhocHRwUHh8gISQlJiUkJikpKiYsoMjd1cfBwsGlaUc5Lzs7P0dNT0JekktnNB8YGBwbHBkXHiEgISIjJCMkJigoLCsmcpO8zKGcq7u4jlJKOjg/LzZGT0JOm095QyMfHh4dHBwZHyEhJSQkIyYlJSUnLS0gkXqhwoqvsnOZfVZJS0k9OTNCSjo9pllzWjEgIB8eHh0aHSEkJyQlJSUkJCUnLSwusbvTqpDUsztbamFUR09APD05PzMxo2VnZTUgHSAeHRwcGyEkJiUmJigkHSEnLCk5y9jWmpzGsISHon5ZUUpIPENAOy4ro4teaT4mFxwbHRwaGSEjICQmKCgkIiEkKSY4xtu8h5jDwbu5qX1VVlFISUQ9OTIomphfZEkpFhscHRwaGSIlHyEkJyckIyMgJikvvNefd43ExMKylGZSVVNKPEw9LSopjp9hbUcpFRsdGx0ZHCIlHxojIiUmIyIdIisiqctmZmm3wbahgVhWWFdMSkUwJCgjhaRlhkc/HRoeGxwZHSAeHBwiIyYlISEeICcchuedZXyxu6yXdVhZWVRWUzIgJygcgq1prU5VJBwcGxsaGx4bGxogJCooJCMgHyUiXcaskaWur5+Kb2JeW09IOSciJCYclslkxFpdLhwdGh0cHCQhIBwgJSonJSQiIicqNqB4fXONo5F8b3BsY1NMPSwlIyMcj9Box3BaRxweHR0aICQkIx4iJSQjJiUjIycsI369hWyVpIl7dHh8Z2CYRjEnICUahNh0vItVVSAdHiAbICUlJB0hJCUlJSYkJCYtJVDYpIWQmIeBgIZ3Xl97WC8zOEMwdMaHsKVPaSkeHx8cIiklJCQmJiYkJCQjIyMpKimh1qeMj4Z/gnRcVlNdeJCgn4xgbbSio7VHdDYhIiAcIykpJiUlJiUjIiEjJCMmKykpYnloUmuBbG2EoL/i6s+UcmhNbberlb9HbUIjIyAbISIjICMiIyUlJCIiIiEgISQkKScnF1j06sGchmpbTkdIQT89NqK+l5x1XX4wJB8dHiAhHRseHiEiIiIhIiAeHiAkJCQfL+PlpaGklmhWRT5CPkJDLqDIq4VzT4VbJiEcHB0dGxsaHyAfHyEhICMhHx8iICYVfeqUp6qdfGBVQDg6QEFDNJfOsoiAWpyNKyAaGhsdHhwaICEcHiIhHyAhHh4dHSMXgKCXp6ief2ROQT4+QkVEMIvTsIyLbtGmNR8ZFxoeGRscHyAfISIiIB8fHB0bHhwrhJmkp6GYj2RHQkJBQUM/KX/LsIqScMa0TR8aFxodGBkbICMfHyAiHhwbHB0dIBhToKKZk5SbmoRMRkE8QEE2Jmy1xIyid7PIWh4bFxsdGBkbHyEiIB8gHB4cHR0dGiZ4lYmHlqiqqJlZUlRHQEE3JXG1xJKGcaGeNx8eGBkaGRcaHSEgHh0cHSAdHhsdGEyQioiVnKSgopZdUl5YUEU6JWmdpJU4JjImICMiGxwbHBoaHR4fHRsbHh0eIB0hG2aSj5qfopuWl5d0VFdUU09DK2uapIg5Ih8jJSUfHB0cHRsbGx0fHBocHR0eHx0hG2iZmqWnoJGPkpOKYU1OTUtOUj4cFh4fISEfIh8ZGxwdHR0bHR4eHBkZHRsfHx8ZS56Xr7SrjoCCg3+HdVFJSERMTU88IiEgICMdIB0XGh0fHh4aGxsbGBMXHB0eHyEgiZudqaOchHl7f3h/ckhDS0tOTkpKKR0hIiMeHRoXGR4fHhwcHBwdGhUVGBweHhpNoJegdX2Mf3ZzdXODcUZBTk1LU0RSOx0gISQdGxoWGRwfHRweHh4gIRsYFx0eHySDmaSaaneKgnVxbXiJeU5CUExJWkVMSyYcHyEcHBwX';const K={neighbors:[1,1,1,1,0,1,1,1,1],average:[1,1,1,1,1,1,1,1,1],laplace:[0,1,0,1,-4,1,0,1,0],sobel:[-1,0,1,-2,0,2,-1,0,1]};function blank(){return Array.from({length:N},()=>Array(N).fill(0))}function copy(a){return a.map(r=>r.slice())}const KB=document.getElementById('kernel');for(let i=0;i<9;i++){let e=document.createElement('input');e.type='number';e.step='.1';KB.appendChild(e)}function setKernel(n){[...KB.children].forEach((e,i)=>e.value=K[n][i]);normalize.checked=n==='average'}function kernel(){return[...KB.children].map(e=>{let v=Number(e.value);if(!Number.isFinite(v))throw Error('Kernel entries must be numbers.');return v})}function idx(i,m){if(m==='wrap')return(i+N)%N;if(m==='reflect')return Math.min(N-1,Math.max(0,i));return i<0||i>=N?null:i}
function perceive(a,r,c,k){let p=0,w=0;for(let dr=-1;dr<=1;dr++)for(let dc=-1;dc<=1;dc++){let q=k[(dr+1)*3+dc+1],rr=idx(r+dr,'wrap'),cc=idx(c+dc,'wrap');p+=q*a[rr][cc];w+=Math.abs(q)}return normalize.checked&&w?p/w:p}function setOf(id){let t=document.getElementById(id).value.trim();if(!t)return[];let a=t.split(/[ ,]+/).filter(Boolean).map(Number);if(a.some(v=>!Number.isFinite(v)))throw Error('Rule values must be numbers.');return a}const clip=v=>Math.min(1,Math.max(0,v));function update(a){let k=kernel(),out=blank(),life=ruleType.value==='life',B,S,F;if(life){B=setOf('born');S=setOf('survive')}else F=new Function('x','p','clip','abs','sin','cos','exp','min','max',`return (${formula.value})`);for(let r=0;r<N;r++)for(let c=0;c<N;c++){let x=a[r][c],p=perceive(a,r,c,k),v;if(life)v=x>.5?+S.some(q=>Math.abs(q-p)<1e-6):+B.some(q=>Math.abs(q-p)<1e-6);else{v=Number(F(x,p,clip,Math.abs,Math.sin,Math.cos,Math.exp,Math.min,Math.max));if(!Number.isFinite(v))throw Error('Formula returned a non-finite value.');if(clamp.checked)v=clip(v)}out[r][c]=v}return out}
function draw(){let d=X.createImageData(N,N);for(let r=0;r<N;r++)for(let c=0;c<N;c++){let v=Math.round(clip(g[r][c])*255),i=(r*N+c)*4;d.data[i]=d.data[i+1]=d.data[i+2]=v;d.data[i+3]=255}let off=document.createElement('canvas');off.width=off.height=N;off.getContext('2d').putImageData(d,0,0);X.imageSmoothingEnabled=false;X.clearRect(0,0,C.width,C.height);X.drawImage(off,0,0,C.width,C.height);let s=C.width/N;X.strokeStyle='rgba(120,120,120,.15)';X.beginPath();for(let i=0;i<=N;i++){X.moveTo(i*s,0);X.lineTo(i*s,C.height);X.moveTo(0,i*s);X.lineTo(C.width,i*s)}X.stroke();stats.textContent=`Generation ${gen}`}function msg(t,bad=false){message.textContent=t||'Click or drag to draw.';message.className=bad?'error':''}function advance(){try{g=update(g);gen++;draw();msg('')}catch(e){pause();msg(e.message,true)}}function schedule(){clearInterval(timer);if(running)timer=setInterval(advance,1000/Number(speed.value))}function pause(){running=false;clearInterval(timer);start.textContent='▶ Start'}function toggle(){running=!running;start.textContent=running?'Ⅱ Pause':'▶ Start';schedule()}
function load(){pause();g=blank();let select=document.getElementById('initial'),p=select.value,m=N>>1;if(p==='Glider')[[0,1],[1,2],[2,0],[2,1],[2,2]].forEach(([r,c])=>g[m-1+r][m-1+c]=1);if(p==='Image'){let bytes=atob(PORTRAIT),rows=bytes.length/N;for(let r=0;r<N;r++)for(let c=0;c<N;c++){let sourceRow=Math.min(rows-1,Math.floor(r*rows/N));g[r][c]=bytes.charCodeAt(sourceRow*N+c)/255}}if(p==='Random binary')for(let r=0;r<N;r++)for(let c=0;c<N;c++)g[r][c]=Math.random()<.25?1:0;if(p==='Random continuous')for(let r=0;r<N;r++)for(let c=0;c<N;c++)g[r][c]=Math.random();if(p==='Single impulse')g[m][m]=1;window.savedInitial=copy(g);gen=0;draw()}function controls(){let b=ruleType.value==='life';life.classList.toggle('hidden',!b);continuous.classList.toggle('hidden',b)}function markCustom(){document.getElementById('updatePreset').value='custom'}function setUpdatePreset(name){if(name==='life'){ruleType.value='life';born.value='3';survive.value='2,3'}else{ruleType.value='formula';if(name==='diffusion')formula.value='x + 0.25 * (p - x)';if(name==='growth')formula.value='clip(x + 0.12 * (p - 0.45))';if(name==='oscillating')formula.value='clip(x + 0.08 * sin(8 * p))';if(name==='tent')formula.value='clip(1 - abs(2 * p - 1))'}controls();msg('')}
function pos(e){let b=C.getBoundingClientRect();return[Math.floor((e.clientY-b.top)*N/b.height),Math.floor((e.clientX-b.left)*N/b.width)]}function paint(e){let[r,c]=pos(e);if(r>=0&&r<N&&c>=0&&c<N){g[r][c]=paintValue;window.savedInitial=copy(g);gen=0;draw()}}C.oncontextmenu=e=>e.preventDefault();C.onpointerdown=e=>{e.preventDefault();pause();drawing=true;paintValue=e.button===2?0:Number(brush.value);C.setPointerCapture(e.pointerId);paint(e)};C.onpointermove=e=>{if(drawing)paint(e)};C.onpointerup=()=>drawing=false;document.getElementById('start').onclick=toggle;document.getElementById('step').onclick=()=>{pause();advance()};document.getElementById('reset').onclick=()=>{pause();g=copy(window.savedInitial);gen=0;draw()};document.getElementById('clear').onclick=()=>{pause();g=blank();window.savedInitial=copy(g);gen=0;draw()};document.getElementById('load').onclick=load;speed.oninput=schedule;brush.oninput=()=>brushLabel.textContent=Number(brush.value).toFixed(2);perceptionPreset.onchange=()=>{setKernel(perceptionPreset.value);draw()};updatePreset.onchange=()=>setUpdatePreset(updatePreset.value);ruleType.onchange=()=>{markCustom();controls()};[...KB.children,normalize].forEach(e=>e.oninput=()=>msg(''));[born,survive,formula,clamp].forEach(e=>e.oninput=()=>{markCustom();msg('')});setKernel('neighbors');normalize.checked=false;setUpdatePreset('life');load();
</script></body></html>
"""
display(HTML(f'<iframe srcdoc="{escape(ca_playground, quote=True)}" width="100%" height="790" style="border:0"></iframe>'))

The playground used handcrafted rules, but its fundamental mechanism was always the same: **perceive locally, update locally, and repeat globally**. Next, we keep this mechanism and replace the handcrafted update rule with a learnable neural network.

## 2. From CA to Neural CA

In the playground, we designed the local rule and observed the resulting global behavior. We now reverse the problem:

> We specify a desired global outcome and train a neural network to discover a local update rule.

This section defines the target and a minimal NCA, trains its local update rule for a few minutes, and inspects the result. This is deliberately a small experiment: the goal is to understand the mechanism, not to maximize performance.

In [ ]:
import json
import random
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image

GRID_SIZE = 48  # Fixed to match the CA playground.
torch.manual_seed(7)
np.random.seed(7)
random.seed(7)
torch.set_num_threads(min(4, torch.get_num_threads()))  # Keeps the tiny CPU workload responsive.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device}")

# Images and pretrained weights live in the tutorial repository. When this notebook runs
# inside a clone of the repository, they are read from disk; otherwise (e.g. in Colab)
# they are downloaded once into ./nca_data.
REPOSITORY = "nca-medical-imaging/tutorial_notebooks"
BRANCH = "session1_ca-to-nca-notebook"  # Change to "main" after merging.
DOWNLOAD_DIR = Path("nca_data")


def fetch(path):
    local = Path(path)
    if local.exists():
        return local
    downloaded = DOWNLOAD_DIR / path
    if not downloaded.exists():
        downloaded.parent.mkdir(parents=True, exist_ok=True)
        url = f"https://raw.githubusercontent.com/{REPOSITORY}/{BRANCH}/{path}"
        urllib.request.urlretrieve(url, downloaded)
    return downloaded

### 2.1 Experiment settings

This is the central place for changing the model and training setup. Start with the defaults, then change one value and rerun the NCA section.

In [ ]:
STATE_CHANNELS = 16         # Four visible channels (RGBA) plus twelve hidden channels.
HIDDEN_SIZE = 64            # Width of the neural update network.
TARGET_PRESET = "lizard"    # Choose: "lizard", "circle", or "cross".

TRAINING_ITERATIONS = 300   # Increase for quality; decrease for a faster experiment.
TRAINING_ROLLOUT_STEPS = 64 # Number of NCA updates before computing the loss.
BATCH_SIZE = 4

# Training ingredients discussed in Section 3. Keep the defaults for the first run.
USE_ALIVE_MASK = False      # True: empty cells far from living cells stay empty.
FIRE_RATE = 1.0             # < 1.0: each cell updates only with this probability per step.

### 2.2 Choose the desired global outcome

The NCA should grow a small lizard emoji from a single cell. Unlike the grayscale image in the playground, the target has four channels: red, green, blue, and **alpha** (opacity). Alpha is 0 in the background, so the NCA has to learn not only *what* to draw but also *where* the pattern ends. The color channels are multiplied by alpha, so empty cells are exactly zero in all four channels.

The simpler circle and cross targets are useful for quick experiments. After changing a setting, rerun the NCA section from this cell onward.

*Emoji: [Noto Emoji](https://github.com/googlefonts/noto-emoji) by Google, Apache License 2.0.*

In [ ]:
EMOJI_SIZE = 40  # The emoji is padded to the grid so there is empty space around it.


def pad_to_grid(rgba):
    canvas = np.zeros((GRID_SIZE, GRID_SIZE, 4), dtype=np.float32)
    offset = (GRID_SIZE - rgba.shape[0]) // 2
    canvas[offset : offset + rgba.shape[0], offset : offset + rgba.shape[1]] = rgba
    return canvas


if TARGET_PRESET == "lizard":
    image = Image.open(fetch("data/emoji_u1f98e.png")).convert("RGBA")
    image = image.resize((EMOJI_SIZE, EMOJI_SIZE), Image.Resampling.LANCZOS)
    rgba = pad_to_grid(np.asarray(image, dtype=np.float32) / 255.0)
else:
    coords = np.linspace(-1, 1, GRID_SIZE)
    y, x = np.meshgrid(coords, coords, indexing="ij")
    if TARGET_PRESET == "circle":
        shape, color = np.sqrt(x**2 + y**2) < 0.6, (0.95, 0.55, 0.1)
    elif TARGET_PRESET == "cross":
        shape, color = (np.abs(x) < 0.14) | (np.abs(y) < 0.14), (0.2, 0.4, 0.9)
        shape &= (np.abs(x) < 0.75) & (np.abs(y) < 0.75)
    else:
        raise ValueError('Choose "lizard", "circle", or "cross".')
    rgba = np.zeros((GRID_SIZE, GRID_SIZE, 4), dtype=np.float32)
    rgba[shape] = (*color, 1.0)

rgba[..., :3] *= rgba[..., 3:]  # Premultiply: color is zero wherever alpha is zero.
target = torch.from_numpy(rgba).permute(2, 0, 1)[None].to(device)


def to_rgb(state):
    # Show the first four channels as color on a white background.
    rgba = state[0, :4].detach().clamp(0, 1).cpu()
    return (rgba[:3] + 1 - rgba[3:4]).clamp(0, 1).permute(1, 2, 0).numpy()


fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(to_rgb(target), interpolation="nearest")
axes[0].set_title(f"Target: {TARGET_PRESET}")
axes[1].imshow(target[0, 3].cpu(), cmap="gray", vmin=0, vmax=1, interpolation="nearest")
axes[1].set_title("Alpha channel")
for ax in axes:
    ax.axis("off")
plt.show()

### 2.3 Build a minimal NCA

In the CA playground, a cell contained only the single value shown on the grid, so its complete state was directly visible. The NCA instead represents every cell by a vector of 16 numbers: the first four are the visible RGBA channels, and the remaining twelve are **hidden channels**.

The hidden channels are not assigned a meaning by us and are not compared with the target during training. They are updated by the same learned local rule as the visible channels and can self-organize into useful intermediate signals, for example "how far am I from the head?". Every cell still communicates only through its local neighborhood.

One NCA step has the familiar structure:

1. `perceive`: apply fixed local filters (identity, Sobel-x, Sobel-y, Laplacian) to every state channel,
2. `update`: process that local perception with a tiny neural network,
3. add the predicted change to the current state.

The update network uses only `1 × 1` convolutions. All spatial communication therefore happens through the explicitly local perception step.

The class also contains two optional ingredients, `alive_mask` and `fire_rate`. They are switched off for the first experiment and explained in Section 3. The `circular` option is used for textures in Section 4.

In [ ]:
ALPHA = 3  # Index of the alpha channel.


class NCA(nn.Module):
    def __init__(self, state_channels, hidden_size, alive_mask=False, fire_rate=1.0, circular=False):
        super().__init__()
        self.state_channels = state_channels
        self.alive_mask = alive_mask
        self.fire_rate = fire_rate
        self.circular = circular  # True: the grid wraps around (used in Section 4).

        identity = torch.tensor([[0, 0, 0], [0, 1, 0], [0, 0, 0]], dtype=torch.float32)
        sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32) / 8
        sobel_y = sobel_x.T
        laplacian = torch.tensor([[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=torch.float32) / 4
        kernels = torch.stack([identity, sobel_x, sobel_y, laplacian])[:, None]
        self.register_buffer("perception_kernels", kernels.repeat(state_channels, 1, 1, 1))

        self.update_net = nn.Sequential(
            nn.Conv2d(4 * state_channels, hidden_size, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(hidden_size, state_channels, kernel_size=1, bias=False),
        )
        nn.init.zeros_(self.update_net[-1].weight)  # The untrained NCA changes nothing.

    def perceive(self, state):
        if self.circular:
            state = F.pad(state, (1, 1, 1, 1), mode="circular")
            return F.conv2d(state, self.perception_kernels, groups=self.state_channels)
        return F.conv2d(state, self.perception_kernels, padding=1, groups=self.state_channels)

    def alive(self, state):
        # A cell is alive if any cell in its 3x3 neighborhood has alpha > 0.1.
        return F.max_pool2d(state[:, ALPHA : ALPHA + 1], 3, stride=1, padding=1) > 0.1

    def forward(self, state):
        pre_alive = self.alive(state) if self.alive_mask else None
        delta = self.update_net(self.perceive(state))
        if self.fire_rate < 1.0:
            delta = delta * (torch.rand_like(state[:, :1]) < self.fire_rate)
        state = state + delta
        if self.alive_mask:
            state = state * (pre_alive & self.alive(state))
        return state


def make_seed(batch_size=1, channels=STATE_CHANNELS):
    # A single cell in the center: invisible color, but alpha and hidden channels set to 1.
    state = torch.zeros(batch_size, channels, GRID_SIZE, GRID_SIZE, device=device)
    state[:, ALPHA:, GRID_SIZE // 2, GRID_SIZE // 2] = 1.0
    return state


model = NCA(STATE_CHANNELS, HIDDEN_SIZE, USE_ALIVE_MASK, FIRE_RATE).to(device)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"State shape: {tuple(make_seed().shape)}")

### 2.4 Train the local update rule

For every optimization step, the NCA starts from the seed and runs for `TRAINING_ROLLOUT_STEPS` updates. Only the four visible channels are compared with the target. Gradients flow back through *all* steps, so the network learns which local update, applied over and over, produces the target. The hidden channels are free to organize useful internal signals.

With the default settings, training takes a few minutes on a CPU.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)
loss_history = []
batch_target = target.expand(BATCH_SIZE, -1, -1, -1)

model.train()
for iteration in range(TRAINING_ITERATIONS + 1):
    state = make_seed(BATCH_SIZE)
    for _ in range(TRAINING_ROLLOUT_STEPS):
        state = model(state)

    loss = F.mse_loss(state[:, :4], batch_target)
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    loss_history.append(loss.item())

    if iteration % 25 == 0:
        print(f"iteration {iteration:3d} | loss {loss.item():.5f}")

### 2.5 Inspect the result

The loss only constrained the state after exactly `TRAINING_ROLLOUT_STEPS` updates. The plot below also shows what happens before and well after that point.

In [ ]:
show_steps = [8, 16, 32, TRAINING_ROLLOUT_STEPS, 2 * TRAINING_ROLLOUT_STEPS, 4 * TRAINING_ROLLOUT_STEPS]

model.eval()
snapshots = {}
with torch.no_grad():
    state = make_seed()
    for step in range(1, max(show_steps) + 1):
        state = model(state)
        if step in show_steps:
            snapshots[step] = torch.nan_to_num(state.clone())

fig, axes = plt.subplots(1, len(show_steps) + 1, figsize=(2.3 * (len(show_steps) + 1), 2.6))
axes[0].plot(loss_history)
axes[0].set(title="Training loss", xlabel="Iteration", yscale="log")
for ax, step in zip(axes[1:], show_steps):
    ax.imshow(to_rgb(snapshots[step]), interpolation="nearest")
    ax.set_title(f"step {step}" + (" (trained)" if step == TRAINING_ROLLOUT_STEPS else ""))
    ax.axis("off")
plt.tight_layout()
plt.show()

**Questions**

- Does the pattern look best at the trained step? What happens afterwards?
- Is the background exactly white, or do faint colors appear outside the lizard?
- The model was only told what the result should look like after a fixed number of steps. What did we *not* ask it to do?

## 3. What makes an NCA stable and able to repair itself?

The minimal NCA usually grows the target, but it was never asked to *stop* growing or to *keep* the pattern. This section compares five pretrained models with the same architecture as above (16 channels, hidden size 64). Each one adds a single training ingredient to the previous one:

| Model | Adds | Idea |
|---|---|---|
| `baseline` | — | The minimal NCA from Section 2, trained longer (2,000 iterations, 64–96 steps). |
| `alive` | alive mask | Cells whose 3×3 neighborhood has no alpha > 0.1 are set to zero. Empty space stays empty, and growth can only spread from living cells. |
| `stochastic` | random updates | Each cell updates with probability 0.5 per step (`FIRE_RATE = 0.5`). There is no global clock, so the rule cannot rely on all cells updating in sync. |
| `pool` | sample pool | Training does not always restart from the seed. Results are stored in a pool and later used as starting points again, so the model must learn to *keep* a finished lizard stable, not only to reach it once. |
| `regenerate` | damage | Some pool samples are partly erased (random circles) before training continues from them, so the model learns to repair. |

The alive mask and random updates are the two options already built into the `NCA` class above. The sample pool and damage only change the training loop. In simplified form:

```python
pool = make_seed(POOL_SIZE)                       # e.g. 512 states
for iteration in range(ITERATIONS):
    idx = random_indices(BATCH_SIZE)
    state = pool[idx]
    state[worst_loss] = make_seed(1)              # keep learning to grow from scratch
    state[best_losses] *= random_circle_masks()   # regenerate only: damage some samples
    for _ in range(random.randint(64, 96)):
        state = model(state)
    loss = mse(state[:, :4], target)
    ...
    pool[idx] = state.detach()                    # results become future starting points
```

The full training code is in `scripts/train_nca_variants.py` in the repository. Training all five took about 30 minutes on a laptop.

In [ ]:
# Same architecture for all variants; only the training ingredients differ.
VARIANTS = {
    "baseline":   dict(alive_mask=False, fire_rate=1.0, use_pool=False, damage=False),
    "alive":      dict(alive_mask=True,  fire_rate=1.0, use_pool=False, damage=False),
    "stochastic": dict(alive_mask=True,  fire_rate=0.5, use_pool=False, damage=False),
    "pool":       dict(alive_mask=True,  fire_rate=0.5, use_pool=True,  damage=False),
    "regenerate": dict(alive_mask=True,  fire_rate=0.5, use_pool=True,  damage=True),
}

pretrained = {}
for name, options in VARIANTS.items():
    checkpoint = torch.load(fetch(f"models/nca_{name}.pt"), map_location=device, weights_only=True)
    variant_model = NCA(16, 64, options["alive_mask"], options["fire_rate"]).to(device)
    variant_model.load_state_dict(checkpoint["state_dict"])
    pretrained[name] = variant_model.eval()
print("Loaded:", ", ".join(pretrained))

### 3.1 Growth, long-term behavior, and damage

The figure runs every model from a single seed for 500 steps. It then grows a fresh lizard for 96 steps, cuts off the head and tail, and lets the model continue. The pretrained models were trained on the lizard, so this figure always uses the lizard target.

In [ ]:
torch.manual_seed(0)
columns = ["step 64", "step 200", "step 500", "grown 96, then cut", "+50 steps", "+150 steps"]
fig, axes = plt.subplots(len(pretrained), len(columns), figsize=(2 * len(columns), 2 * len(pretrained)))

with torch.no_grad():
    for row, (name, variant_model) in enumerate(pretrained.items()):
        images = []
        state = make_seed(channels=16)
        for step in range(1, 501):
            state = variant_model(state)
            if step in (64, 200, 500):
                images.append(to_rgb(torch.nan_to_num(state)))

        state = make_seed(channels=16)
        for _ in range(96):
            state = variant_model(state)
        state[:, :, 32:, 26:] = 0   # cut the tail
        state[:, :, :20, :22] = 0   # cut the head
        images.append(to_rgb(state))
        for step in range(1, 151):
            state = variant_model(state)
            if step in (50, 150):
                images.append(to_rgb(torch.nan_to_num(state)))

        for col, image in enumerate(images):
            axes[row, col].imshow(image, interpolation="nearest")
            axes[row, col].set_xticks([])
            axes[row, col].set_yticks([])
            if row == 0:
                axes[row, col].set_title(columns[col], fontsize=9)
        axes[row, 0].set_ylabel(name)
plt.tight_layout()
plt.show()

**Questions**

- Which models are still a lizard at step 500? Which explode, and which slowly smear out?
- Compare `baseline` and `alive` at step 64. What does the alive mask fix, and what does it not fix?
- `pool` was never trained with damage. Why can it still partly repair the cut?
- Why might random updates (`stochastic`) make the rule more robust, even though they make each rollout noisier?

### 3.2 Explore the models interactively

The app below runs the model you trained in Section 2 and all five pretrained models directly in the browser. You can follow the NCA one step at a time, damage it while it runs, and inspect the hidden channels, the alive cells, and the update magnitude.

**Tasks**

1. Select **baseline**, load a single seed, and press **Start**. Watch it pass step 100. Then repeat with **pool**.
2. Switch the visualization to **Alive cells** and compare **baseline** and **alive**. Which cells does each model update?
3. Select **stochastic** and use **Step**. How does growth look different when only half of the cells update per step?
4. Let **pool** and **regenerate** grow a lizard. With the **Erase** tool, cut off the tail or head while the model runs. Try large cuts, and cuts through the middle. Which model recovers, and how quickly?
5. Load **Two seeds** with **regenerate**. What happens when two growing lizards meet?
6. Inspect the hidden channels during regrowth. Can you find a channel that looks like a "body map", a gradient, or a boundary signal?
7. Select **Update magnitude** for a fully grown **regenerate** lizard. Is it still changing? What happens right after damage?

**If time allows:** change one setting in Section 2.1 (for example `USE_ALIVE_MASK = True`, `FIRE_RATE = 0.5`, or a different target), rerun Sections 2.2–2.5 and this app, and compare the **Just trained** model with the pretrained ones.

In [ ]:
#@title NCA app code (HTML/JavaScript)
NCA_APP = r"""<!doctype html><html><head><meta charset="utf-8"><style>
*{box-sizing:border-box}body{margin:0;font-family:system-ui,sans-serif;color:#202124}.app{max-width:980px;margin:auto;padding:14px;border:1px solid #dadce0;border-radius:14px;background:#f8f9fa}.top,.row,.stats{display:flex;gap:8px;align-items:center;flex-wrap:wrap}.top{margin-bottom:12px}.layout{display:grid;grid-template-columns:minmax(340px,1fr) 270px;gap:14px}.canvas-wrap{background:#222;border-radius:10px;padding:8px;display:flex}.canvas-wrap canvas{width:100%;aspect-ratio:1;cursor:crosshair;touch-action:none}.card{background:#fff;border:1px solid #dadce0;border-radius:10px;padding:12px;margin-bottom:10px}.card h3{font-size:15px;margin:0 0 9px}button,select,input{font:inherit;border:1px solid #bdc1c6;border-radius:7px;background:#fff;padding:6px 8px}button{cursor:pointer;font-weight:600}.primary{background:#333;color:#fff!important}.wide{width:100%}label{font-size:13px;font-weight:600}.small{font-size:12px;color:#5f6368;line-height:1.4}.stats{justify-content:space-between;margin-top:8px;font-size:13px}.tags{display:flex;gap:4px;flex-wrap:wrap;margin-top:8px}.tag{font-size:11px;padding:2px 6px;border-radius:9px;background:#e8eaed;color:#5f6368}.tag.on{background:#1e8e3e;color:#fff}input[type=range]{padding:0}@media(max-width:720px){.layout{grid-template-columns:1fr}}
</style></head><body><div class="app"><div class="top"><button id="start" class="primary">▶ Start</button><button id="step">Step</button><button id="reset">Reset</button><button id="clear">Clear</button><label>Speed <input id="speed" type="range" min="1" max="30" value="15"></label></div><div class="layout"><div><div class="canvas-wrap"><canvas id="canvas" width="640" height="640"></canvas></div><div class="stats"><span id="generation">Generation 0</span><span id="hint"></span></div></div><div>
<div class="card"><h3>Model</h3><select id="modelChoice" class="wide"></select><div id="modelDescription" class="small" style="margin-top:8px"></div><div id="tags" class="tags"></div></div>
<div class="card"><h3>Initial state</h3><div class="row"><select id="initialPreset" style="flex:1"><option value="center">Single seed</option><option value="two">Two seeds</option><option value="four">Four seeds</option><option value="noise">Random noise</option><option value="empty">Empty grid</option></select><button id="load">Load</button></div></div>
<div class="card"><h3>Mouse tool</h3><select id="tool" class="wide"><option value="damage">Erase (damage)</option><option value="seed">Place seed</option></select><label style="display:block;margin-top:8px">Erase radius <span id="radiusLabel"></span><input id="radius" class="wide" type="range" min="1" max="12" value="6"></label><div class="small">Right-click always erases.</div></div>
<div class="card"><h3>Perception angle</h3><input id="angle" class="wide" type="range" min="-180" max="180" step="15" value="0"><div class="small"><span id="angleLabel">0°</span> · rotates the Sobel filters only. The network is unchanged.</div></div>
<div class="card"><h3>Visualization</h3><select id="channel" class="wide"></select><div id="description" class="small" style="margin-top:8px"></div></div>
</div></div></div>
<script>
const M=__MODEL__,N=M.size,P=N*N,canvas=document.getElementById('canvas'),ctx=canvas.getContext('2d'),ALPHA=3;
let W,C,H,state,initial,lastDelta,generation=0,running=false,timer=null,drawing=false;const at=(c,i)=>c*P+i;
for(const[key,m]of Object.entries(M.models))modelChoice.add(new Option(m.label,key));
function configureModel(){W=M.models[modelChoice.value];C=W.channels;H=W.hidden_size;state=new Float32Array(C*P);lastDelta=new Float32Array(C*P);
 modelDescription.textContent=W.description||'';
 tags.innerHTML=(W.circular?[['texture loss',true],['wrap-around grid',true],['random updates',W.fire_rate<1],['sample pool',W.use_pool]]:[['alive mask',W.alive_mask],['random updates',W.fire_rate<1],['sample pool',W.use_pool],['damage',W.damage]]).map(([t,on])=>`<span class="tag ${on?'on':''}">${on?'✓':'✗'} ${t}</span>`).join('');
 let keep=channel.value,V=W.circular?3:4;channel.innerHTML=`<option value="rgba">Visible (${W.circular?'RGB':'RGBA'})</option><option value="target">Training target</option>`+(W.alive_mask?'<option value="alive">Alive cells</option>':'')+'<option value="delta">Update magnitude</option>'+Array.from({length:C-V},(_,i)=>`<option value="${i+V}">Hidden channel ${i+1}</option>`).join('');channel.value=[...channel.options].some(o=>o.value===keep)?keep:'rgba'}
function seed(r,c){if(r<0||r>=N||c<0||c>=N)return;for(let ch=0;ch<C;ch++)state[at(ch,r*N+c)]=ch>=ALPHA?1:0}
function erase(r0,c0,rad){for(let r=Math.max(0,r0-rad);r<=Math.min(N-1,r0+rad);r++)for(let c=Math.max(0,c0-rad);c<=Math.min(N-1,c0+rad);c++)if((r-r0)**2+(c-c0)**2<=rad*rad)for(let ch=0;ch<C;ch++)state[at(ch,r*N+c)]=0}
function loadInitial(){pause();state=new Float32Array(C*P);let p=initialPreset.value,m=Math.floor(N/2);if(p==='center')seed(m,m);if(p==='two'){seed(m,m-12);seed(m,m+12)}if(p==='four'){seed(m-12,m-12);seed(m-12,m+12);seed(m+12,m-12);seed(m+12,m+12)}if(p==='noise')for(let i=0;i<state.length;i++)state[i]=(Math.random()-0.5)*0.5;initial=new Float32Array(state);lastDelta.fill(0);generation=0;draw()}
function aliveMask(s){let a=new Uint8Array(P);for(let r=0;r<N;r++)for(let c=0;c<N;c++){let mx=-Infinity;for(let dr=-1;dr<=1;dr++)for(let dc=-1;dc<=1;dc++){let rr=r+dr,cc=c+dc;if(rr>=0&&rr<N&&cc>=0&&cc<N)mx=Math.max(mx,s[at(ALPHA,rr*N+cc)])}a[r*N+c]=mx>0.1?1:0}return a}
function update(){
 const th=Number(angle.value)*Math.PI/180,cs=Math.cos(th),sn=Math.sin(th),K=W.kernels,pre=W.alive_mask?aliveMask(state):null;
 let perc=new Float32Array(C*4*P);
 for(let ch=0;ch<C;ch++)for(let r=0;r<N;r++)for(let c=0;c<N;c++){let v=[0,0,0,0];for(let dr=-1;dr<=1;dr++)for(let dc=-1;dc<=1;dc++){let rr=r+dr,cc=c+dc;if(W.circular){rr=(rr+N)%N;cc=(cc+N)%N}else if(rr<0||rr>=N||cc<0||cc>=N)continue;let s=state[at(ch,rr*N+cc)],k=(dr+1)*3+dc+1;for(let q=0;q<4;q++)v[q]+=K[q][k]*s}
  let i=r*N+c;perc[(ch*4)*P+i]=v[0];perc[(ch*4+1)*P+i]=cs*v[1]-sn*v[2];perc[(ch*4+2)*P+i]=sn*v[1]+cs*v[2];perc[(ch*4+3)*P+i]=v[3]}
 let next=new Float32Array(state),hidden=new Float32Array(H),w1=W.weight_1,b1=W.bias_1,w2=W.weight_2;lastDelta.fill(0);
 for(let i=0;i<P;i++){if(W.fire_rate<1&&Math.random()>=W.fire_rate)continue;
  for(let h=0;h<H;h++){let z=b1[h],row=w1[h];for(let j=0;j<C*4;j++)z+=row[j]*perc[j*P+i];hidden[h]=z>0?z:0}
  for(let ch=0;ch<C;ch++){let d=0,row=w2[ch];for(let h=0;h<H;h++)d+=row[h]*hidden[h];lastDelta[ch*P+i]=d;next[ch*P+i]+=d}}
 if(W.alive_mask){let post=aliveMask(next);for(let i=0;i<P;i++)if(!(pre[i]&&post[i]))for(let ch=0;ch<C;ch++)next[ch*P+i]=0}
 state=next;generation++;draw()}
const clamp=x=>Math.max(0,Math.min(1,x));
function rgbaOverWhite(get,i){let a=clamp(get(3,i));return[0,1,2].map(ch=>Math.round(255*clamp(get(ch,i)+1-a)))}
function drawTexture(){let T=M.texture_size,img=ctx.createImageData(T,T);for(let i=0;i<T*T;i++){for(let ch=0;ch<3;ch++)img.data[i*4+ch]=255*M.texture[ch*T*T+i];img.data[i*4+3]=255}let off=document.createElement('canvas');off.width=off.height=T;off.getContext('2d').putImageData(img,0,0);ctx.imageSmoothingEnabled=false;ctx.drawImage(off,0,0,canvas.width,canvas.height);generationEl.textContent=`Generation ${generation}`;description.textContent=`Example of the target texture (${T}×${T}). The NCA matches its statistics, not its pixels.`}
function draw(){let sel=channel.value,img=ctx.createImageData(N,N),px;
 if(sel==='rgba'&&W.circular)px=i=>[0,1,2].map(ch=>255*clamp(state[at(ch,i)]+0.5));
 else if(sel==='rgba')px=i=>rgbaOverWhite((ch,j)=>state[at(ch,j)],i);
 else if(sel==='target'&&W.circular){drawTexture();return}
 else if(sel==='target')px=i=>rgbaOverWhite((ch,j)=>M.target[ch*P+j],i);
 else if(sel==='alive'){let a=aliveMask(state);px=i=>a[i]?[30,142,62]:[240,240,240]}
 else if(sel==='delta'){let mag=new Float32Array(P);for(let i=0;i<P;i++){let s=0;for(let ch=0;ch<C;ch++)s+=lastDelta[ch*P+i]**2;mag[i]=Math.sqrt(s)}let sc=Math.max(...mag,1e-6);px=i=>{let q=clamp(mag[i]/sc);return[255*q,180*q*q,30*q*q]}}
 else{let ch=Number(sel),sc=1e-6;for(let i=0;i<P;i++)sc=Math.max(sc,Math.abs(state[at(ch,i)]));px=i=>{let v=state[at(ch,i)],q=clamp(Math.abs(v)/sc);return v>=0?[255,255*(1-q),255*(1-q)]:[255*(1-q),255*(1-q),255]}}
 for(let i=0;i<P;i++){let c=px(i),j=i*4;img.data[j]=c[0];img.data[j+1]=c[1];img.data[j+2]=c[2];img.data[j+3]=255}
 let off=document.createElement('canvas');off.width=off.height=N;off.getContext('2d').putImageData(img,0,0);ctx.imageSmoothingEnabled=false;ctx.drawImage(off,0,0,canvas.width,canvas.height);
 generationEl.textContent=`Generation ${generation}`;
 description.textContent={rgba:W.circular?'Channels 0–2 shown as RGB (0 = mid-gray).':'Channels 0–3 shown as color over a white background.',target:'What the visible channels were trained to match.',alive:'Green cells are alive (alpha > 0.1 in their 3×3 neighborhood). Only these cells are updated when the model uses an alive mask.',delta:'Magnitude of the most recent update across all channels.'}[sel]||'Hidden channel (blue = negative, red = positive; range normalized per frame).'}
function pause(){running=false;clearInterval(timer);start.textContent='▶ Start'}function toggle(){running=!running;start.textContent=running?'Ⅱ Pause':'▶ Start';clearInterval(timer);if(running)timer=setInterval(update,1000/Number(speed.value))}
function position(e){let b=canvas.getBoundingClientRect();return[Math.floor((e.clientY-b.top)*N/b.height),Math.floor((e.clientX-b.left)*N/b.width)]}
function paint(e){let[r,c]=position(e);if(e.buttons===2||tool.value==='damage')erase(r,c,Number(radius.value));else{seed(r,c);initial=new Float32Array(state);generation=0}draw()}
const generationEl=document.getElementById('generation');canvas.oncontextmenu=e=>e.preventDefault();
canvas.onpointerdown=e=>{e.preventDefault();drawing=true;canvas.setPointerCapture(e.pointerId);paint(e)};canvas.onpointermove=e=>{if(drawing&&(e.buttons===2||tool.value==='damage'))paint(e)};canvas.onpointerup=()=>drawing=false;
start.onclick=toggle;step.onclick=()=>{pause();update()};reset.onclick=()=>{pause();state=new Float32Array(initial);lastDelta.fill(0);generation=0;draw()};clear.onclick=()=>{pause();state.fill(0);initial=new Float32Array(state);lastDelta.fill(0);generation=0;draw()};
document.getElementById('load').onclick=loadInitial;modelChoice.onchange=()=>{configureModel();if(W.circular)initialPreset.value='empty';else if(initialPreset.value==='empty'||initialPreset.value==='noise')initialPreset.value='center';loadInitial()};channel.onchange=draw;speed.oninput=()=>{if(running){pause();toggle()}};
radius.oninput=()=>radiusLabel.textContent=radius.value;angle.oninput=()=>angleLabel.textContent=angle.value+'°';tool.onchange=()=>hint.textContent=tool.value==='damage'?'Drag to erase cells while it runs':'Click to place a seed';
radius.oninput();tool.onchange();configureModel();loadInitial();
</script></body></html>"""

In [ ]:
from html import escape
from IPython.display import HTML, display

MODEL_INFO = {
    "live": ("Just trained (Section 2)", "The model you trained above."),
    "baseline": ("baseline", "Synchronous updates, no alive mask, always trained from the seed."),
    "alive": ("alive", "+ alive mask: cells far from living cells stay empty."),
    "stochastic": ("stochastic", "+ random updates: each cell updates with probability 0.5 per step."),
    "pool": ("pool", "+ sample pool: training continues from earlier results, so the lizard must stay stable."),
    "regenerate": ("regenerate", "+ damage: some pool samples are cut before training, so the model learns to repair."),
    "texture": ("texture (Section 4)", "Trained to match the statistics of a spot texture. Starts from an empty grid; edges wrap around."),
}


def export_model(name, selected_model, options):
    label, description = MODEL_INFO[name]
    return {
        "label": label,
        "description": description,
        **options,
        "circular": selected_model.circular,
        "channels": selected_model.state_channels,
        "hidden_size": selected_model.update_net[0].out_channels,
        "kernels": selected_model.perception_kernels[:4, 0].flatten(1).cpu().tolist(),
        "weight_1": selected_model.update_net[0].weight[:, :, 0, 0].detach().cpu().tolist(),
        "bias_1": selected_model.update_net[0].bias.detach().cpu().tolist(),
        "weight_2": selected_model.update_net[2].weight[:, :, 0, 0].detach().cpu().tolist(),
    }



def show_nca_app():
    live_options = dict(alive_mask=USE_ALIVE_MASK, fire_rate=FIRE_RATE, use_pool=False, damage=False)
    models_for_app = {"live": export_model("live", model, live_options)}
    for name, variant_model in pretrained.items():
        models_for_app[name] = export_model(name, variant_model, VARIANTS[name])
    payload = {"size": GRID_SIZE, "target": target[0].cpu().flatten().tolist(), "models": models_for_app}
    if "texture_model" in globals():  # Added once Section 4 has been run.
        texture_options = dict(alive_mask=False, fire_rate=0.5, use_pool=True, damage=False)
        models_for_app["texture"] = export_model("texture", texture_model, texture_options)
        payload["texture"] = texture_target[0].cpu().flatten().tolist()
        payload["texture_size"] = texture_target.shape[-1]
    app = NCA_APP.replace("__MODEL__", json.dumps(payload))
    display(HTML(f'<iframe srcdoc="{escape(app, quote=True)}" width="100%" height="900" style="border:0"></iframe>'))


show_nca_app()

### 3.3 Rotating the perception

The **Perception angle** slider rotates the Sobel-x and Sobel-y filters before they are used. The learned network is not changed at all.

**Tasks**

1. Set the angle to 90°, load a single seed with **regenerate**, and let it grow. What do you see?
2. Try 45°. Is the result as clean as at 90°?

**Discussion:** The Sobel filters tell each cell in which direction a channel increases, which gives it a sense of "up" and "right". The identity and Laplacian filters look the same in every direction. Why does rotating only the two gradient filters rotate the whole pattern? Why might 45° work less well on a square grid?

## 4. Beyond a single image: a texture NCA

So far, the NCA had to reproduce one specific image, pixel by pixel. The same architecture can also learn a **texture**: a pattern that should look similar everywhere but has no fixed position, size, or end. The target below is a reaction-diffusion pattern of spots, the kind of pattern that also appears on animal skin.

Compared with the lizard models, four things change:

| | Lizard (Sections 2–3) | Texture |
|---|---|---|
| Start | one seed cell | an empty grid; random updates break the symmetry |
| Loss | pixel-wise MSE to the target image | difference in **texture statistics**: which VGG16 features occur together, regardless of where (Gram matrices, as in neural style transfer) |
| Grid | edges | wraps around (`circular=True`), so there is no border |
| Goal | grow a shape, then stop | keep producing the texture forever |

Because the loss ignores *where* features are, the NCA cannot copy the example. It has to learn a local rule that keeps creating new spots wherever there is space. The same rule then works on any grid size.

This model has 12 channels (RGB + 9 hidden) and a hidden size of 96. It was trained for 3,000 iterations with a sample pool in about 8 minutes on a laptop GPU. The training code is in `scripts/train_texture_nca.py`. It needs the pretrained VGG16 network, so we only load the result here.

In [ ]:
texture_model = NCA(12, 96, fire_rate=0.5, circular=True).to(device)
checkpoint = torch.load(fetch("models/nca_texture.pt"), map_location=device, weights_only=True)
texture_model.load_state_dict(checkpoint["state_dict"])
texture_model.eval()
texture_image = np.asarray(Image.open(fetch("models/texture_target.png")).convert("RGB"))
texture_target = torch.from_numpy(texture_image / 255.0).float().permute(2, 0, 1)[None]


def texture_rgb(state):
    # Texture NCAs show channels 0-2 shifted by 0.5, so an empty grid is mid-gray.
    return (state[0, :3].detach().cpu() + 0.5).clamp(0, 1).permute(1, 2, 0).numpy()


def run_texture(size, steps, damage_at=None):
    state = torch.zeros(1, 12, size, size, device=device)
    frames = {}
    with torch.no_grad():
        for step in range(1, max(steps) + 1):
            state = texture_model(state)
            if step == damage_at:
                state[:, :, size // 4 : size // 2, size // 4 : size // 2] = 0
            if step in steps:
                frames[step] = texture_rgb(state)
    return frames


torch.manual_seed(0)
small = run_texture(48, [10, 30, 60, 120, 300])
damaged = run_texture(48, [301, 330, 400], damage_at=300)
large = run_texture(128, [300])

fig, axes = plt.subplots(2, 5, figsize=(13, 5.6))
axes[0, 0].imshow(texture_image, interpolation="nearest")
axes[0, 0].set_title("Example texture")
for ax, (step, frame) in zip(axes[0, 1:], list(small.items())[1:]):
    ax.imshow(frame, interpolation="nearest")
    ax.set_title(f"48×48, step {step}")
axes[1, 0].imshow(small[10], interpolation="nearest")
axes[1, 0].set_title("48×48, step 10")
for ax, (step, frame) in zip(axes[1, 1:4], damaged.items()):
    ax.imshow(frame, interpolation="nearest")
    ax.set_title("damaged at 300" if step == 301 else f"step {step}")
axes[1, 4].imshow(large[300], interpolation="nearest")
axes[1, 4].set_title("128×128, step 300")
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

**Questions**

- The model was trained on a 64×64 grid. Why does the same rule work on a 128×128 grid?
- Does the damaged region come back exactly as before? Should it?
- Compare the texture loss with the lizard loss. What kind of information does each one ignore?

### 4.1 Explore the texture interactively

Running the cell below shows the app again, now with the **texture** model included.

**Tasks**

1. Select **texture**. It starts from an empty (gray) grid. Press **Start** and watch how the spots appear. Where do the first structures form?
2. Erase a region while it runs. How does the texture fill the hole?
3. Load **Random noise** as the initial state. Does the result differ from the empty start?
4. Inspect the hidden channels. Do any of them look like the spots, and are some shifted or blurred versions?
5. Erase a stripe along the left edge of the grid. Do the spots near the border behave differently from those in the center? Why not?

In [ ]:
show_nca_app()

## 5. Take-away

- Repeated **local interactions and updates** can create coherent global behavior without centralized control.
- With deep learning, we do not have to design the local rule by hand. We can specify the desired global outcome and optimize a shared local update rule with gradient descent through many steps.
- What the NCA learns depends strongly on **how it is trained**. A model trained only to reach the target once may explode afterwards. An alive mask, random updates, a sample pool, and damage during training lead to patterns that are stable and can repair themselves.
- The update rule only sees local gradients, so changing how a cell perceives its neighborhood, for example rotating the filters, changes the global result without retraining.
- The loss defines what "success" means. Matching texture statistics instead of pixels turns the same architecture into a texture generator that runs forever, on any grid size.
- Visualizing hidden channels and updates over time offers insight into how the NCA organizes and transforms information.